# 0.9 · 概率论 / Probability

> **课程定位 / Where this fits**
> 第 9 课，**Part 0 · 基础准备**。
> Lesson 9, **Part 0 · Foundations**.
>
> 数据科学的所有"**不确定性**"、"**推断**"、"**生成**" 都建立在概率论上：朴素贝叶斯、GMM、贝叶斯优化、变分推断、扩散模型、LLM 的下一个 token——全是概率。
> Everything in DS about uncertainty / inference / generation lives here: Naive Bayes, GMM, Bayesian opt, VI, diffusion models, LLM next-token prediction — all probability.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $\Pr(A)$ —— 事件 $A$ 的概率 / probability of event $A$
> - $p(x)$ —— 概率密度 / probability density
> - $X \sim \mathcal{N}(\mu, \sigma^2)$ —— $X$ 服从正态分布
> - $\mathbb{E}[X]$ —— 期望 / expectation
> - $\mathrm{Var}(X)$, $\mathrm{Cov}(X, Y)$, $\boldsymbol{\Sigma}$ —— 方差 / 协方差 / 协方差矩阵
> - $\mathbb{1}\{A\}$ —— 指示函数 / indicator

> 💡 **面试相关 / Interview-relevant**
> - Bayes 公式应用题（疾病诊断、垃圾邮件）—— 高频
> - "什么是 MLE / MAP" —— 必考
> - 解释 CLT 为什么有用 —— 高频
> - 期望/方差的代数操作（$\mathbb{E}[aX+b]$, $\mathrm{Var}(X+Y)$）—— 必考
>
> Frequent interview hits: Bayes apps, MLE vs MAP, CLT, expectation/variance algebra.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 用**概率三公理**严格推出条件概率 + 全概率 + 贝叶斯公式。
   Derive conditional / total prob / Bayes from the three axioms.
2. 区分**离散** vs **连续**随机变量的 PMF / PDF / CDF。
   Distinguish PMF / PDF / CDF for discrete vs continuous RVs.
3. 自信地用**期望与方差的线性性**做代数操作。
   Manipulate expectation and variance algebraically.
4. 写出 **10 种核心分布**的 PMF/PDF、期望、方差、典型用途。
   Know PMF/PDF, mean, var, and uses of the **10 core distributions**.
5. 解释**独立 vs 不相关**的差别（独立 ⇒ 不相关，反之不成立）。
   Tell apart independence and uncorrelatedness.
6. 解释**中心极限定理**为什么是 DS 的"魔法"。
   Explain why CLT is foundational.
7. 推导一个 **MLE** 例子（伯努利 + 正态）并跟梯度下降对照。
   Derive MLE for Bernoulli and Normal and contrast with GD.
8. 在 **SMS Spam** 数据上手写**朴素贝叶斯**完成垃圾邮件分类。
   Build **Naive Bayes** spam classifier from scratch on **SMS Spam**.

---

## 目录 / Table of Contents

1. [样本空间、事件、公理 / Sample Space & Axioms](#1)
2. [条件概率 & 贝叶斯 / Conditional & Bayes ⭐](#2)
3. [独立性 / Independence](#3)
4. [随机变量 / Random Variables (PMF / PDF / CDF)](#4)
5. [期望与方差 / Expectation & Variance](#5)
6. [10 大核心分布 / The 10 Core Distributions](#6)
7. [联合分布、协方差、相关性 / Joint, Cov, Correlation](#7)
8. [多元正态 / Multivariate Normal](#8)
9. [大数定律 & **中心极限定理** ⭐ / LLN & CLT](#9)
10. [最大似然估计 / MLE](#10)
11. [实战：手写 Naive Bayes 做 SMS 垃圾邮件分类 / Hands-on](#11)
12. [小结 / Summary](#12)


<a id="1"></a>
## 1. 样本空间、事件、公理 / Sample Space & Axioms

### 基本定义 / Basic definitions

- **样本空间** $\Omega$：一次随机实验**所有可能结果的集合**。
  Sample space = the set of all possible outcomes.
- **事件** $A \subseteq \Omega$：$\Omega$ 的子集（"可能发生的某件事"）。
  Event = a subset of $\Omega$.

例：扔一颗骰子 → $\Omega = \{1,2,3,4,5,6\}$，"扔出偶数" = $\{2, 4, 6\}$。

### 概率三公理（Kolmogorov）/ Kolmogorov axioms

对每个事件 $A$，概率 $\Pr(A)$ 满足：

1. **非负** / Non-negativity: $\Pr(A) \ge 0$
2. **归一** / Normalization: $\Pr(\Omega) = 1$
3. **可数可加** / Countable additivity: 若 $A_1, A_2, \dots$ 两两不交，则
   $$\Pr\bigl(\bigcup_i A_i\bigr) = \sum_i \Pr(A_i)$$

### 由公理立刻推出 / Immediate consequences

- $\Pr(\varnothing) = 0$
- $\Pr(A^c) = 1 - \Pr(A)$
- $\Pr(A \cup B) = \Pr(A) + \Pr(B) - \Pr(A \cap B)$（容斥）


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st
import pandas as pd

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

# 用模拟来"验证"公理直觉 / Simulate to confirm intuition
n_throws = 100_000
throws = rng.integers(1, 7, size=n_throws)        # 1..6

# P(even) = 1/2
p_even = (throws % 2 == 0).mean()
# P(prime in {2,3,5}) = 3/6 = 1/2
p_prime = np.isin(throws, [2, 3, 5]).mean()
# P(even AND prime) = P(2) = 1/6
p_even_and_prime = ((throws % 2 == 0) & np.isin(throws, [2, 3, 5])).mean()
# Check inclusion-exclusion / 容斥
p_either = (((throws % 2 == 0) | np.isin(throws, [2, 3, 5]))).mean()
expected = p_even + p_prime - p_even_and_prime

print(f"P(even)         ≈ {p_even:.4f}   (true 0.5)")
print(f"P(prime)        ≈ {p_prime:.4f}   (true 0.5)")
print(f"P(even ∩ prime) ≈ {p_even_and_prime:.4f}   (true 1/6 ≈ 0.1667)")
print(f"P(even ∪ prime) ≈ {p_either:.4f}")
print(f"by inclusion-excl: {expected:.4f}")


<a id="2"></a>
## 2. 条件概率 & 贝叶斯 ⭐ / Conditional Probability & Bayes

### 2.1 条件概率 / Conditional probability

$$
\Pr(A \mid B) = \frac{\Pr(A \cap B)}{\Pr(B)} \quad (\Pr(B) > 0)
$$

"已知 $B$ 发生时，$A$ 发生的概率"。
"Probability of $A$ given $B$ has occurred."

### 2.2 链式法则 / Chain rule

$$\Pr(A_1 \cap \dots \cap A_n) = \Pr(A_1)\Pr(A_2 \mid A_1)\Pr(A_3 \mid A_1, A_2)\cdots$$

### 2.3 全概率公式 / Law of total probability

如果 $\{B_i\}$ 是 $\Omega$ 的一个**划分**：

$$\Pr(A) = \sum_i \Pr(A \mid B_i)\Pr(B_i)$$

### 2.4 贝叶斯公式 ⭐ / Bayes' Rule

由条件概率 + 全概率立刻推出：

$$
\boxed{\;\Pr(B_i \mid A) = \dfrac{\Pr(A \mid B_i)\,\Pr(B_i)}{\Pr(A)} = \dfrac{\Pr(A \mid B_i)\,\Pr(B_i)}{\sum_j \Pr(A \mid B_j)\,\Pr(B_j)}\;}
$$

| 名字 / Name | 含义 / Meaning |
|---|---|
| $\Pr(B_i)$ | **先验** / prior |
| $\Pr(A \mid B_i)$ | **似然** / likelihood |
| $\Pr(B_i \mid A)$ | **后验** / posterior |
| $\Pr(A)$ | **证据**（归一化常数）/ evidence |

> 💡 **机器学习一句话总览 / One-line ML overview**
> 频率派看到的是似然 $\Pr(\text{data}\mid\text{params})$，找最大化它的参数（**MLE**）；
> 贝叶斯派算后验 $\Pr(\text{params}\mid\text{data})$。
> Frequentists maximize the likelihood (MLE); Bayesians compute the posterior.

### 经典例题 / Classic example: disease testing

某病在普通人群中的发病率是 1%。有一种检测，**真阳率** 99%、**假阳率** 5%。
A disease affects 1% of the general population. Test sensitivity 99%, false-positive rate 5%.

**问题**：你检测呈阳性，你真的有病的概率是多少？
**Question**: you tested positive — what's the probability you actually have it?


In [ ]:
# 解：用 Bayes / Solve via Bayes
P_D = 0.01                    # P(disease) = 先验 / prior
P_pos_given_D = 0.99          # 真阳率 / sensitivity
P_pos_given_noD = 0.05        # 假阳率 / false-positive rate
P_noD = 1 - P_D

# Bayes:  P(D | pos) = P(pos | D) P(D) / P(pos)
P_pos = P_pos_given_D * P_D + P_pos_given_noD * P_noD
P_D_given_pos = (P_pos_given_D * P_D) / P_pos

print(f"P(disease | test positive) = {P_D_given_pos:.4f}  ≈ {P_D_given_pos*100:.1f}%")


**答案：≈ 16.7%** —— 比大多数人直觉小得多！
**Answer: ~16.7%** — far smaller than most people guess.

为什么？因为**先验 1% 太低了**：群里没病的人是有病人的 99 倍，即使每个没病的人只有 5% 假阳率，**假阳性总数依然 5× 真阳性**。
Because the base rate is tiny — even with low false-positive rate, false positives dominate.

这就是 Bayes 在医学、风控、推荐系统里至关重要的原因——**别忘了先验**。
This is why Bayes matters in medicine, risk, and recommenders — **always account for the prior**.


<a id="3"></a>
## 3. 独立性 / Independence

### 定义 / Definition

$A, B$ **独立**（$A \perp\!\!\!\perp B$）当且仅当：

$$\Pr(A \cap B) = \Pr(A)\,\Pr(B)$$

或等价地：$\Pr(A \mid B) = \Pr(A)$（"$B$ 给不出关于 $A$ 的信息"）。

### 多事件 / Multi-event

$A_1, \dots, A_n$ **相互独立** ⇔ 任意子集积都等于积。
Pairwise independence is weaker than joint independence — needs all sub-products.

### 条件独立 / Conditional independence

$A \perp\!\!\!\perp B \mid C$：
$$\Pr(A \cap B \mid C) = \Pr(A \mid C)\,\Pr(B \mid C)$$

> **条件独立**是**朴素贝叶斯** + **贝叶斯网**的核心假设。
> Conditional independence is the cornerstone of Naive Bayes and Bayesian networks.


<a id="4"></a>
## 4. 随机变量 / Random Variables

**随机变量 $X$** = 把样本空间映射到实数的函数 $X: \Omega \to \mathbb{R}$。
A random variable maps sample space to reals.

例：扔骰子，$X$ = 点数。

### 4.1 离散：PMF / Discrete: probability mass function

$$p_X(x) = \Pr(X = x)$$

性质：$\sum_x p_X(x) = 1$，$p_X(x) \in [0, 1]$。

### 4.2 连续：PDF / Continuous: probability density function

对于**连续** $X$，单点概率为 0，要用密度：
For continuous RVs, point probabilities are zero; use density:

$$\Pr(a \le X \le b) = \int_a^b p_X(x)\,dx$$

**❗注意**：$p_X(x)$ **不是概率**，可以 > 1。它代表"概率每单位长度"。
**Caveat**: $p_X(x)$ is **not a probability** — can exceed 1 (it's per unit length).

### 4.3 CDF（两类都有）/ Cumulative Distribution Function

$$F_X(x) = \Pr(X \le x)$$

性质：单调递增，$F_X(-\infty) = 0$，$F_X(+\infty) = 1$。

### 三者关系 / Relations

- 离散：$F_X(x) = \sum_{x' \le x} p_X(x')$
- 连续：$F_X(x) = \int_{-\infty}^x p_X(t)\,dt$，$\;p_X(x) = F_X'(x)$


<a id="5"></a>
## 5. 期望与方差 / Expectation & Variance

### 5.1 期望 / Expectation

- 离散：$\mathbb{E}[X] = \sum_x x\,p_X(x)$
- 连续：$\mathbb{E}[X] = \int x\,p_X(x)\,dx$
- 函数：$\mathbb{E}[g(X)] = \sum_x g(x)\,p_X(x)$ （或积分）

### 5.2 期望的关键性质 / Key properties

| 性质 / Property | 公式 / Formula |
|---|---|
| 线性性 / Linearity | $\mathbb{E}[aX + bY + c] = a\mathbb{E}[X] + b\mathbb{E}[Y] + c$ ⭐ |
| 独立 → 积 / Indep ⇒ product | $X \perp\!\!\!\perp Y \Rightarrow \mathbb{E}[XY] = \mathbb{E}[X]\mathbb{E}[Y]$ |
| 不变 / Constant | $\mathbb{E}[c] = c$ |

⭐ 线性性**不要求独立**，这是它最强大的地方。
⭐ Linearity does **not** require independence — that's why it's so powerful.

### 5.3 方差 / Variance

$$\mathrm{Var}(X) = \mathbb{E}[(X - \mathbb{E}[X])^2] = \mathbb{E}[X^2] - (\mathbb{E}[X])^2$$

### 5.4 方差的关键性质 / Key properties

| 性质 / Property | 公式 / Formula |
|---|---|
| 平移不变 / Shift invariance | $\mathrm{Var}(X + c) = \mathrm{Var}(X)$ |
| 缩放 / Scaling | $\mathrm{Var}(aX) = a^2\,\mathrm{Var}(X)$ |
| 独立和 / Indep sum | $X \perp\!\!\!\perp Y \Rightarrow \mathrm{Var}(X + Y) = \mathrm{Var}(X) + \mathrm{Var}(Y)$ ⭐ |
| **一般和 / General sum** | $\mathrm{Var}(X + Y) = \mathrm{Var}(X) + \mathrm{Var}(Y) + 2\,\mathrm{Cov}(X, Y)$ |

> ⭐ **独立和的方差相加** + 线性性 → **中心极限定理**和**蒙特卡洛**的基础。


In [ ]:
# 验证：模拟一个简单 RV，计算期望和方差
# X = 骰子点数 / Die face
n_throws = 1_000_000
X = rng.integers(1, 7, size=n_throws)

E_X = X.mean()
Var_X = X.var(ddof=0)

# 理论：E[X] = (1+2+...+6)/6 = 3.5
# Var[X] = E[X²] - E[X]² = (1²+...+6²)/6 - 3.5²
# = 91/6 - 12.25 = 15.1667 - 12.25 = 2.9167
print(f"E[X]   sim: {E_X:.4f}    true: 3.5")
print(f"Var[X] sim: {Var_X:.4f}    true: ≈ 2.9167")


<a id="6"></a>
## 6. 10 大核心分布 / The 10 Core Distributions

DS 里 95% 的统计场景都涵盖在这 10 个分布里。**每个都要知道：PMF/PDF、期望、方差、典型用途**。
95% of DS stats lives in these 10. **For each: know PMF/PDF, mean, var, when to use.**

| 分布 / Distribution | 类型 | $\mathbb{E}[X]$ | $\mathrm{Var}(X)$ | 用例 / Use |
|---|---|---|---|---|
| Bernoulli $\mathrm{Ber}(p)$ | 离散 | $p$ | $p(1-p)$ | 单次二分类 / one binary trial |
| Binomial $\mathrm{Bin}(n,p)$ | 离散 | $np$ | $np(1-p)$ | $n$ 次独立二分类 / repeated trials |
| Geometric $\mathrm{Geom}(p)$ | 离散 | $1/p$ | $(1-p)/p^2$ | 首次成功前的失败次数 |
| Poisson $\mathrm{Poi}(\lambda)$ | 离散 | $\lambda$ | $\lambda$ | 事件计数（呼叫到达数）|
| Categorical | 离散 | — | — | softmax 输出 / one of $K$ |
| Uniform $\mathcal{U}(a,b)$ | 连续 | $(a+b)/2$ | $(b-a)^2/12$ | 无信息 / random init |
| Normal $\mathcal{N}(\mu, \sigma^2)$ | 连续 | $\mu$ | $\sigma^2$ | **王者**：CLT、误差、隐变量 |
| Exponential $\mathrm{Exp}(\lambda)$ | 连续 | $1/\lambda$ | $1/\lambda^2$ | 等待时间 |
| Gamma $\Gamma(\alpha, \beta)$ | 连续 | $\alpha/\beta$ | $\alpha/\beta^2$ | Exp 的推广、Beta 的 sister |
| Beta $\mathrm{Beta}(\alpha, \beta)$ | 连续 | $\alpha/(\alpha+\beta)$ | (omit) | 概率的先验、A/B 测试 |


In [ ]:
# 一张图把 10 个分布全画了 / All 10 distributions in one figure
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
ax = axes.flatten()

# Discrete: bar plots
# 1) Bernoulli(0.3)
k = np.arange(0, 2)
ax[0].bar(k, st.bernoulli(0.3).pmf(k))
ax[0].set_title("Bernoulli(p=0.3)\n E=0.3, Var=0.21"); ax[0].set_xticks([0, 1])

# 2) Binomial(10, 0.4)
k = np.arange(0, 11)
ax[1].bar(k, st.binom(10, 0.4).pmf(k))
ax[1].set_title("Binomial(n=10, p=0.4)\n E=4, Var=2.4")

# 3) Geometric(0.3)  (number of failures before first success — scipy convention)
k = np.arange(0, 15)
ax[2].bar(k, st.geom(0.3).pmf(k + 1))
ax[2].set_title("Geometric(p=0.3)\n trials to first success")

# 4) Poisson(3)
k = np.arange(0, 15)
ax[3].bar(k, st.poisson(3).pmf(k))
ax[3].set_title("Poisson(λ=3)\n E=3, Var=3")

# 5) Categorical(K=4 probs)
ax[4].bar(np.arange(4), [0.1, 0.4, 0.3, 0.2])
ax[4].set_title("Categorical (K=4)\n e.g. softmax output")
ax[4].set_xticks(range(4))

# Continuous: line plots
# 6) Uniform(-1, 1)
x = np.linspace(-2, 2, 300)
ax[5].plot(x, st.uniform(-1, 2).pdf(x))    # scipy uses loc=-1, scale=2
ax[5].set_title("Uniform(-1, 1)\n E=0, Var=1/3")

# 7) Normal(0, 1)
x = np.linspace(-4, 4, 300)
ax[6].plot(x, st.norm.pdf(x))
ax[6].set_title("Normal(0, 1)\n E=0, Var=1")

# 8) Exponential(λ=1)
x = np.linspace(0, 5, 300)
ax[7].plot(x, st.expon.pdf(x))
ax[7].set_title("Exponential(λ=1)\n E=1, Var=1")

# 9) Gamma(α=2, β=1)
x = np.linspace(0, 8, 300)
ax[8].plot(x, st.gamma(2).pdf(x))
ax[8].set_title("Gamma(α=2, β=1)\n E=2, Var=2")

# 10) Beta(2, 5)
x = np.linspace(0, 1, 300)
ax[9].plot(x, st.beta(2, 5).pdf(x))
ax[9].set_title("Beta(α=2, β=5)\n E=2/7, supported on [0,1]")

for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 💡 工作中怎么挑分布 / Which distribution when

| 你看到 / You see | 第一反应 / First guess |
|---|---|
| 二选一标签 / binary label | Bernoulli |
| 一段时间内事件计数 / count in interval | Poisson |
| 等待时间 / waiting time | Exponential |
| 误差、误差和、CLT 适用 | **Normal** |
| 概率本身的先验 / prior over a probability | Beta |
| 比率/方差先验 / prior over rate/variance | Gamma |
| 类别 softmax | Categorical |

> ⭐ **Beta 是 Bernoulli/Binomial 的共轭先验**，A/B 测试做贝叶斯分析的标配。
> Beta is the conjugate prior of Bernoulli/Binomial — the workhorse of Bayesian A/B testing.


<a id="7"></a>
## 7. 联合分布、协方差、相关性 / Joint, Cov, Correlation

### 7.1 联合分布 / Joint distribution

二元 RV $(X, Y)$ 的**联合 PMF/PDF** 描述两者**同时**的概率行为：
$$\Pr(X = x, Y = y) \quad\text{或}\quad p_{X,Y}(x, y)$$

### 7.2 边缘分布 / Marginal

$$p_X(x) = \sum_y p_{X,Y}(x, y) \quad\text{(or integral)}$$

### 7.3 条件分布 / Conditional

$$p_{Y\mid X}(y \mid x) = \frac{p_{X,Y}(x, y)}{p_X(x)}$$

### 7.4 协方差 / Covariance

$$\mathrm{Cov}(X, Y) = \mathbb{E}\bigl[(X - \mathbb{E}[X])(Y - \mathbb{E}[Y])\bigr] = \mathbb{E}[XY] - \mathbb{E}[X]\mathbb{E}[Y]$$

性质：
- $\mathrm{Cov}(X, X) = \mathrm{Var}(X)$
- $\mathrm{Cov}(aX + b, cY + d) = ac\,\mathrm{Cov}(X, Y)$
- **$X \perp\!\!\!\perp Y \Rightarrow \mathrm{Cov}(X, Y) = 0$**（反向不一定！）

### 7.5 (Pearson) 相关系数 / Correlation

$$\rho_{X,Y} = \frac{\mathrm{Cov}(X, Y)}{\sigma_X \sigma_Y} \in [-1, 1]$$

仅刻画**线性关系**。
Captures only **linear** dependence.

> ⚠ **独立 ⇒ 不相关；反过来不成立**。
> Independence implies uncorrelated; converse fails.


In [ ]:
# 反例：X 和 Y = X² 是高度依赖的，但相关系数 ≈ 0
# Counter-example: X and Y = X² are dependent but uncorrelated
rng2 = np.random.default_rng(0)
X = rng2.uniform(-1, 1, size=10_000)
Y = X**2

rho = np.corrcoef(X, Y)[0, 1]
print(f"correlation(X, X²) = {rho:.4f}    (expect ~0)")

# 但他们显然不独立：知道 X 完全确定 Y
# Yet clearly NOT independent: knowing X tells you Y exactly
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X[:2000], Y[:2000], alpha=0.3, s=8)
ax.set_xlabel("X"); ax.set_ylabel("Y = X²")
ax.set_title(f"Uncorrelated (ρ={rho:.3f}) but NOT independent")
plt.show()


<a id="8"></a>
## 8. 多元正态 / Multivariate Normal $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$

**机器学习里最重要的多元分布**——线性回归误差假设、Kalman 滤波、GMM、贝叶斯线性、扩散模型……
**The single most important multivariate distribution in ML.**

### 密度 / Density

$$p(\mathbf{x}) = \frac{1}{(2\pi)^{d/2}\,|\boldsymbol{\Sigma}|^{1/2}}\,\exp\!\left(-\tfrac{1}{2}(\mathbf{x} - \boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1}(\mathbf{x} - \boldsymbol{\mu})\right)$$

- $\boldsymbol{\mu} \in \mathbb{R}^d$ —— 均值向量 / mean
- $\boldsymbol{\Sigma} \in \mathbb{R}^{d\times d}$ —— **对称正定**协方差矩阵 / positive-definite covariance

### 关键事实 / Key facts

1. **任意线性组合仍是正态** / Linear combinations are normal:
   $\mathbf{A}\mathbf{X} + \mathbf{b} \sim \mathcal{N}(\mathbf{A}\boldsymbol{\mu} + \mathbf{b},\; \mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^\top)$
2. **边缘 / 条件还是正态** / Marginals and conditionals are normal
3. **不相关 = 独立**（**仅在正态情形下！**）/ For Gaussians only, uncorrelated ⇔ independent
4. 等概率线 = **椭圆**，主轴 = 协方差矩阵的特征向量
   Iso-density lines are ellipses with axes along $\boldsymbol{\Sigma}$'s eigenvectors

性质 4 把上节线性代数和概率连起来了！PCA = 找椭圆主轴 = 协方差特征向量。
Property 4 ties linear algebra to probability — PCA finds the ellipse axes.


In [ ]:
# 多元正态采样 + 椭圆可视化 / Sample + ellipse viz
mu = np.array([1.0, 2.0])
Sigma = np.array([[2.0, 1.2],
                  [1.2, 1.0]])

samples = rng.multivariate_normal(mu, Sigma, size=2000)

# 用谱分解算等概率椭圆 / Compute iso-density ellipse via spectral decomp
eigvals, eigvecs = np.linalg.eigh(Sigma)
print(f"eigvals: {eigvals}")
print(f"eigvecs (columns) = main ellipse axes:\n{eigvecs}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(samples[:, 0], samples[:, 1], alpha=0.25, s=8)

# 画 1σ, 2σ 椭圆 / Plot 1σ, 2σ ellipses
theta = np.linspace(0, 2*np.pi, 200)
unit = np.array([np.cos(theta), np.sin(theta)])             # 2 × 200
for k_sigma in [1, 2]:
    # 把单位圆变换：mu + k * eigvecs @ diag(sqrt(eigvals)) @ unit
    L = eigvecs @ np.diag(np.sqrt(eigvals))
    ellipse = mu[:, None] + k_sigma * L @ unit
    ax.plot(ellipse[0], ellipse[1], color="red", lw=2,
            label=f"{k_sigma}σ ellipse" if k_sigma == 1 else None)

# 把椭圆主轴也画出来 / Draw principal axes
for i in range(2):
    vec = eigvecs[:, i] * np.sqrt(eigvals[i]) * 2
    ax.plot([mu[0] - vec[0], mu[0] + vec[0]],
            [mu[1] - vec[1], mu[1] + vec[1]], "g-", lw=2)
ax.scatter(mu[0], mu[1], color="black", s=80, zorder=5, label="μ")
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("Bivariate normal samples + iso-density ellipses\n(green = principal axes from Σ eigen)")
plt.show()


<a id="9"></a>
## 9. 大数定律 & 中心极限定理 ⭐ / LLN & CLT

设 $X_1, X_2, \dots, X_n$ i.i.d.，$\mu = \mathbb{E}[X_i]$，$\sigma^2 = \mathrm{Var}(X_i) < \infty$。

### 9.1 大数定律 / Law of large numbers (LLN)

$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i \;\xrightarrow{p}\; \mu$$

样本均值"收敛"到总体均值。**蒙特卡洛**就是基于这条定理。

### 9.2 中心极限定理 / Central Limit Theorem ⭐

$$\sqrt{n}\,(\bar{X}_n - \mu) \;\xrightarrow{d}\; \mathcal{N}(0,\,\sigma^2)$$

或等价地：
$$\bar{X}_n \;\overset{\text{approx}}{\sim}\; \mathcal{N}\!\left(\mu,\,\dfrac{\sigma^2}{n}\right) \;\;\text{for large } n$$

**人话翻译**：**不管 $X_i$ 来自什么分布**（只要方差有限），只要 $n$ 够大，**它们的平均/和都近似正态分布**。
**In words**: **regardless of the underlying distribution** (with finite variance), large-$n$ averages are approximately normal.

### 这就是为什么正态分布在 DS 里"无处不在"
- 噪声叠加 → 正态（电子学、测量误差、股票收益短期）
- A/B 测试的均值差 → 正态（即使原数据是漏斗转化）
- 抽样分布 → 正态（大样本统计推断的基础）


In [ ]:
# 演示 CLT：从指数分布抽样取均值 / Empirical CLT from exponential samples
def sample_means(n_per_avg, n_samples=5000):
    # 从 Exp(1) 抽 n_per_avg 个数，求均值，重复 n_samples 次
    return rng.exponential(scale=1.0, size=(n_samples, n_per_avg)).mean(axis=1)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, n in zip(axes, [1, 2, 10, 100]):
    means = sample_means(n)
    ax.hist(means, bins=50, density=True, alpha=0.7, color="steelblue")
    # 理论正态：均值 = 1，方差 = 1/n
    x = np.linspace(means.min(), means.max(), 200)
    ax.plot(x, st.norm(loc=1, scale=np.sqrt(1/n)).pdf(x), "r-", lw=2,
            label=f"N(1, 1/{n})")
    ax.set_title(f"avg of {n} Exp(1)")
    ax.legend(fontsize=9)
plt.suptitle("Central Limit Theorem in action — averages approach Gaussian", y=1.05)
plt.tight_layout()
plt.show()


**左边**：单个指数样本 → 强烈右偏 → 不是正态
**右边**：100 个指数样本的平均 → 几乎完美贴合 $\mathcal{N}(1, 0.01)$

The leftmost panel is heavily right-skewed (exponential); by $n=100$ the average is indistinguishable from $\mathcal{N}(1, 0.01)$.

> 💡 **面试一句话答 / One-line interview**
> "CLT lets us treat any large-sample mean (or sum) as approximately Gaussian, which is why z-tests / t-tests / confidence intervals work even when the underlying distribution is unknown."


<a id="10"></a>
## 10. 最大似然估计 / MLE

观察到数据 $\mathcal{D} = \{x_1, \dots, x_n\}$，假设它们 i.i.d. 来自参数化的分布族 $p(x \mid \boldsymbol{\theta})$。

**似然函数 / Likelihood**：
$$\mathcal{L}(\boldsymbol{\theta}; \mathcal{D}) = \prod_{i=1}^n p(x_i \mid \boldsymbol{\theta})$$

**对数似然 / Log-likelihood**：
$$\ell(\boldsymbol{\theta}) = \log \mathcal{L}(\boldsymbol{\theta}; \mathcal{D}) = \sum_{i=1}^n \log p(x_i \mid \boldsymbol{\theta})$$

**MLE**：
$$\hat{\boldsymbol{\theta}}_{\mathrm{MLE}} = \arg\max_{\boldsymbol{\theta}} \ell(\boldsymbol{\theta})$$

> **机器学习几乎所有的"训练 loss"都是负对数似然**：
> Most ML losses are negative log-likelihood.
> - 回归 MSE = 在高斯噪声假设下的负 log-likelihood
> - 二分类 cross-entropy = 伯努利似然
> - softmax cross-entropy = 类别分布的似然

### 例 1：Bernoulli MLE

数据：$x_1, \dots, x_n \in \{0, 1\}$，$p$ 是参数。

$$\ell(p) = \sum_i [x_i \log p + (1 - x_i)\log(1 - p)] = k \log p + (n-k)\log(1-p)$$
其中 $k = \sum_i x_i$。

求导置 0：$\dfrac{k}{p} - \dfrac{n-k}{1-p} = 0 \Rightarrow \hat{p} = k/n$（频率！）

### 例 2：Normal MLE（$\mu, \sigma^2$）

可推导（见经典统计书）：
$$\hat{\mu} = \bar{x}, \qquad \hat{\sigma}^2 = \frac{1}{n}\sum_i(x_i - \bar{x})^2$$

注意 **MLE 的方差是 $1/n$（有偏），无偏估计是 $1/(n-1)$**——下一节统计里详谈。
**MLE variance is biased (uses $1/n$); the unbiased version uses $1/(n-1)$.**


In [ ]:
# 数值验证：从 N(2, 3²) 抽 1000 个样本，MLE 估 μ, σ²
true_mu, true_sigma = 2.0, 3.0
data = rng.normal(true_mu, true_sigma, size=1000)

mu_mle = data.mean()
sigma2_mle = ((data - mu_mle)**2).mean()       # 1/n version
sigma2_unbiased = data.var(ddof=1)             # 1/(n-1) version

print(f"true   μ   = {true_mu},      σ² = {true_sigma**2:.4f}")
print(f"MLE    μ̂  = {mu_mle:.4f},   σ̂² = {sigma2_mle:.4f}    (biased, /n)")
print(f"unbias μ̂  = {mu_mle:.4f},   σ̂² = {sigma2_unbiased:.4f}  (unbiased, /(n-1))")


<a id="11"></a>
## 11. 实战：手写 Naive Bayes 做 SMS 垃圾邮件分类 / Hands-on

把本节所有工具串起来：分类概率 + 贝叶斯 + 条件独立 + MLE。
Tie everything together: class probabilities + Bayes + conditional indep + MLE.

> **📱 数据集介绍 / Dataset Intro: SMS Spam Collection**
>
> **来源 / Source**: UCI ML Repository。Tiago Almeida 等人 2011 年发布。
> UCI ML Repo; published 2011.
>
> **内容 / Contents**: 5,574 条**真实**英文短信，每条标 `ham`（正常）或 `spam`（垃圾）。
> 5,574 real English SMS messages labeled `ham` or `spam`.
>
> **任务 / Task**: 二分类——判断一条 SMS 是不是垃圾。
> Binary classification: spam vs. ham.
>
> **为什么经典 / Why classic**: 自然语言文本 + 不平衡（约 13% 垃圾）+ 简单标注 → **Naive Bayes 教学的标准集**。
> Real text + class imbalance (~13% spam) + clean labels — the textbook NB dataset.
>
> 我们这里**自己构造**一份小数据集做演示（直接 `sklearn` 也有 20-Newsgroups 可换）。
> We'll use a tiny in-notebook synthetic version (full dataset usually needs download).

### Naive Bayes 数学

对一条短信 $\mathbf{x} = (w_1, w_2, \dots, w_d)$（**词袋**表示，$w_j \in \{0, 1\}$ 表示词 $j$ 是否出现），类别 $y \in \{\text{spam}, \text{ham}\}$。

后验：
$$\Pr(y \mid \mathbf{x}) = \frac{\Pr(\mathbf{x} \mid y)\,\Pr(y)}{\Pr(\mathbf{x})}$$

**朴素 / Naive 假设**：给定类别 $y$，词独立。
**Naive assumption**: words are conditionally independent given the class.
$$\Pr(\mathbf{x} \mid y) = \prod_{j=1}^d \Pr(w_j \mid y)$$

预测规则（去掉公共的 $\Pr(\mathbf{x})$）：
$$\hat{y} = \arg\max_y\; \log\Pr(y) + \sum_j \log\Pr(w_j \mid y)$$

MLE 估参数 + **拉普拉斯平滑**避免 $\log 0$：
$$\hat{\Pr}(w_j = 1 \mid y) = \dfrac{\text{count}(w_j = 1, y) + 1}{\text{count}(y) + 2}$$


In [ ]:
# 构造一个 mini SMS 数据集 / Tiny in-notebook SMS dataset
# 真实使用：sklearn.datasets.fetch_openml('SMSSpamCollection') 或下 UCI
data = pd.DataFrame({
    "text": [
        # spam
        "WIN a free iPhone today click here now",
        "URGENT you have won a 1000 dollar prize call now",
        "Free entry into our weekly prize draw text WIN to 12345",
        "Congratulations you have been selected for a free vacation",
        "Click this link to claim your free reward immediately",
        "Limited offer act now to claim free gift card",
        "You have won cash prize call this number now",
        "Free ringtones text the number to win",
        # ham
        "Hey are we still meeting for lunch tomorrow",
        "Can you pick up some milk on your way home",
        "Happy birthday hope you have a great day",
        "I will be late to the meeting sorry",
        "Did you watch the game last night",
        "Lets grab coffee this weekend if you are free",
        "Mom called she wants you to call back",
        "Running errands now will see you at home tonight",
    ],
    "label": ["spam"] * 8 + ["ham"] * 8,
})
print(data.head())
print(f"\nclass balance:\n{data['label'].value_counts()}")


In [ ]:
# 构建词表 / Build vocab + Bernoulli BoW features
from collections import Counter
import re

def tokenize(text): return re.findall(r"[a-z]+", text.lower())

vocab = sorted({w for t in data["text"] for w in tokenize(t)})
print(f"vocab size: {len(vocab)}")

def featurize(text):
    # Bernoulli bag-of-words: 1 if word present, else 0
    words = set(tokenize(text))
    return np.array([1 if w in words else 0 for w in vocab])

X = np.vstack([featurize(t) for t in data["text"]])
y = (data["label"] == "spam").astype(int).values        # 1 = spam
print(f"X shape: {X.shape}    (n_samples × vocab_size)")


In [ ]:
# Naive Bayes 训练 = 数频次 + 平滑 / Train: count + smooth
def train_nb(X, y, alpha=1.0):
    # Train Bernoulli Naive Bayes; return log-priors and log-likelihoods
    n, d = X.shape
    classes = np.unique(y)
    log_prior = np.zeros(len(classes))
    log_lik = np.zeros((len(classes), d))       # log P(w_j = 1 | y)
    log_lik_neg = np.zeros((len(classes), d))   # log P(w_j = 0 | y)

    for c in classes:
        X_c = X[y == c]
        n_c = X_c.shape[0]
        log_prior[c] = np.log(n_c / n)
        # Laplace smoothing: (count + α) / (n_c + 2α)
        p_w_given_c = (X_c.sum(axis=0) + alpha) / (n_c + 2 * alpha)
        log_lik[c] = np.log(p_w_given_c)
        log_lik_neg[c] = np.log(1 - p_w_given_c)
    return log_prior, log_lik, log_lik_neg

log_prior, log_lik, log_lik_neg = train_nb(X, y)
print(f"log priors: {log_prior.round(3)}")
print(f"P(class)  : {np.exp(log_prior).round(3)}")


In [ ]:
# 预测 / Predict
def predict_nb(X_new, log_prior, log_lik, log_lik_neg):
    # log P(y | x) ∝ log P(y) + Σ_j [x_j log P(w_j=1|y) + (1-x_j) log P(w_j=0|y)]
    log_post = (
        log_prior +
        X_new @ log_lik.T +
        (1 - X_new) @ log_lik_neg.T
    )
    return log_post.argmax(axis=1)

# 训练集准确率（演示用，不严格）/ Train accuracy (demo only)
y_pred = predict_nb(X, log_prior, log_lik, log_lik_neg)
acc = (y_pred == y).mean()
print(f"train accuracy: {acc:.3f}")

# 测试新短信 / Predict on new messages
new = [
    "WIN a free vacation click here",          # 显然 spam
    "Will you join us for dinner tonight",     # 显然 ham
    "Free meeting tomorrow at 3pm",            # 模糊 / ambiguous
]
X_new = np.vstack([featurize(t) for t in new])
preds = predict_nb(X_new, log_prior, log_lik, log_lik_neg)
classes = ["ham", "spam"]
for text, p in zip(new, preds):
    print(f"  [{classes[p]}]  {text}")


**短短 30 行代码**就把 Bayes + 条件独立 + MLE + 平滑全用上了 —— 这是机器学习"最经济"的算法之一。
**30 lines covers Bayes + cond. independence + MLE + smoothing** — one of the most economical ML algorithms.

> 💡 **Naive Bayes 实战注意 / Real-world tips**
> - **`log` 概率求和**而不是原始概率相乘 —— 否则 underflow
> - **Laplace 平滑 ($\alpha=1$)** 几乎是默认 —— 避免训练集里没出现的词把概率推到 0
> - 文本分类基线，**至今仍是垃圾邮件、文档语言识别的强 baseline**


<a id="12"></a>
## 12. 小结 / Summary

### 概念地图 / Concept map

```
概率公理
  │
  ├── 条件概率 ──→ Bayes 公式 ⭐
  │                    │
  │                    └── MLE / MAP / Bayesian inference
  │
  ├── 独立性 → 条件独立 → Naive Bayes / 贝叶斯网
  │
  └── 随机变量 (PMF / PDF / CDF)
        │
        ├── 期望 E[X] —— 线性性 ⭐
        ├── 方差 Var(X), Cov(X,Y), ρ
        │
        ├── 离散分布：Bernoulli, Binomial, Poisson, Geom, Categorical
        ├── 连续分布：Uniform, Normal, Exp, Gamma, Beta
        │
        ├── 联合 → 边缘 / 条件
        ├── 多元正态 → 椭圆 → 谱分解 → PCA
        │
        └── n 个 i.i.d.
              │
              ├── LLN → 蒙特卡洛
              └── CLT ⭐ → 正态、z/t 检验、CI
```

### 🧠 必记公式 / Must-know formulas

| 概念 | 公式 |
|---|---|
| Bayes | $\Pr(B \mid A) = \dfrac{\Pr(A \mid B)\Pr(B)}{\Pr(A)}$ |
| 全概率 | $\Pr(A) = \sum_i \Pr(A\mid B_i)\Pr(B_i)$ |
| 期望线性 | $\mathbb{E}[aX+bY+c] = a\mathbb{E}[X]+b\mathbb{E}[Y]+c$ |
| 方差缩放 | $\mathrm{Var}(aX) = a^2 \mathrm{Var}(X)$ |
| 独立和方差 | $\mathrm{Var}(X+Y) = \mathrm{Var}(X)+\mathrm{Var}(Y)$ |
| 一般和方差 | $\mathrm{Var}(X+Y) = \mathrm{Var}(X)+\mathrm{Var}(Y)+2\mathrm{Cov}(X,Y)$ |
| CLT | $\bar{X}_n \approx \mathcal{N}(\mu, \sigma^2/n)$ |
| Bernoulli MLE | $\hat{p} = k/n$ |
| Normal MLE | $\hat{\mu}=\bar{x}$, $\hat{\sigma}^2 = \frac{1}{n}\sum(x_i - \bar{x})^2$ |

### 💡 工业速查 / Industry cheat sheet

```python
import scipy.stats as st

# 各种分布的 PDF/PMF/CDF/采样 / Common ops
X = st.norm(loc=0, scale=1)
X.pdf(0); X.cdf(1.96); X.ppf(0.975); X.rvs(1000)

# MLE 估正态参数（带 numpy）/ Fit normal
mu, sigma = data.mean(), data.std(ddof=1)

# 朴素贝叶斯（sklearn）/ Naive Bayes
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB

# 多元正态 / Multivariate normal
st.multivariate_normal(mean=mu, cov=Sigma).pdf(point)

# CLT 用 z/t test
from scipy.stats import ttest_ind
t, p = ttest_ind(group_a, group_b)
```

### 💡 面试速查 / Interview must-knows

1. **Bayes 应用题先识别"先验+似然+证据"** 三件套
2. **CLT 一句话总结**：任何"足够多"独立样本的均值都近似正态
3. **MLE vs MAP**：MAP = MLE + 先验（拉格朗日意义下的正则化）
4. **独立 ⇒ 不相关 ✓；不相关 ⇒ 独立 ✗（除非高斯）**
5. **熟记 10 大分布**的 $\mathbb{E}, \mathrm{Var}$
6. **Naive Bayes 假设是"条件独立"** — 工程上虽然几乎不成立，但很多时候够用

### 下一节预告 / Next up

**Part 0.10 · 数值优化** —— 凸性、KKT 条件、Newton 法、L-BFGS、Adam 谱系。把梯度下降"升级"为现代优化器。
**Part 0.10 · Numerical Optimization** — convexity, KKT, Newton, L-BFGS, the Adam family. Upgrade GD into modern optimizers.
